# Validation v3 · the elevation maps of a maps-only site

Santander and Santoña (the design `experiments/validation_v3_cantabrian_2026-10-29.md`, its amendment of 2026-11-03):
their PNOA LiDAR was flown at high water, so they get elevation maps and no score. The cells up to the estimates are
notebook 05's, cell for cell (`NB_V3` names `santander_maps` or `santona_maps`: no truth, no class mask, the window of
the 100 screened scenes nearest 2024-07-15, every pixel of the production's no-delay product); the last cells write
the maps. The elevations are on the boundary's datum (EOT20 + the nearest IOC gauge), as every product of the project.

## Parameters

Everything that changes between runs lives in the next cell. The environment variables `NB_SITE`, `NB_NS`,
`NB_METHODS`, `NB_EXTRA`, `NB_OUT`, `NB_NTABLES`, `NB_FIRST`, `NB_BED` and `NB_ROW_YEARS` override them (the server
script sets `NB_SITE`). The confirmatory windows: `NB_NS=125,100,75`, with `NB_OUT` a folder of their own and
`NB_NTABLES` notebook 01's run of the same windows. The constants of the last block are notebook 01's.

In [ ]:
import os

SITE    = os.environ.get("NB_SITE", "escalda")       # escalda | wadden | ems
NS      = os.environ.get("NB_NS")                    # the windows, e.g. "60,50" or "125,100,75"; unset: the site's
                                                     # development pair (next cell)
METHODS = os.environ.get("NB_METHODS", "MAREA,MAREA_ndwi0,MAREA_B,MAREA_BL,MAREA_B_nodelay,MAREA_nodelay,"
                                       "E4,E4_delay,E5c,DEA,NIDEM,E5a").split(",")
EXTRA   = os.environ.get("NB_EXTRA", "auto")         # pre-2023 scenes, as notebook 01 reads them ("auto", "none", a path)
OUT     = os.environ.get("NB_OUT", "results/attribution_nb")        # the CSVs, the npz and the figures go here
NTABLES = os.environ.get("NB_NTABLES", "results/fixedcfg_nb_diag")  # notebook 01 with its variants: the last check
FIRST   = os.environ.get("NB_FIRST", "results/fixedcfg_nb")         # ... else notebook 01's first run (nine methods)
BED     = os.environ.get("NB_BED", "results/bed_change_nb")         # notebook 04's change class per pixel (optional)
ROW_YEARS = [int(v) for v in os.environ.get("NB_ROW_YEARS", "2025").split(",")]   # the survey-year rows analysed
ONE_ROW = os.environ.get("NB_ONE_ROW", "1") == "1"   # 1: one row per site (the Wadden's 2023-2025 surveys together)
V3KEY   = os.environ.get("NB_V3")                    # a validation v3 truth (experiments/v3/sites.py, e.g. foz_2016):
                                                     # its cube, PNOA truth, symmetric windows and class mask; unset:
                                                     # the Dutch sites as before
Q_CLASS = float(os.environ.get("NB_Q", "0.10"))      # v3: a pixel is scored when q <= f_wet <= 1 - q in the window
Q_SENS  = (0.0, 0.05, 0.10, 0.15, 0.20)              # v3: the sensitivity of the scores to q (the design)

# ── The analysis ────────────────────────────────────────────────────────
Q_BINS      = 5           # quantile bins of the strata (section 5); the positive delays get 4 (section 6)
GALLERY     = (12, 4)     # the pixels where V0 errs most / least relative to LogiTide2DEM (section 7)
B_MIN_LOOKS = 8           # MAREA-B: at least 8 clear looks, one wet and one dry (the variants file)
B_TIE_TOL   = 1e-9        # MAREA-B: z levels whose profile log-likelihood is within this of the maximum are tied
B_LAPSES    = (0.0, 0.02, 0.05, 0.10)   # V10: the lapse rates e, maximised with z and sigma (the variants file)

# ── Fixed by notebook 01 ────────────────────────────────────────────────
MAX_PX            = 200_000   # a larger F is v19's seeded random subsample
SEED              = 20261019
RANGE_PAD_M       = 0.25      # truth kept inside the production no-delay range ± 0.25 m
MAREA_MIN_BRACKET = 0         # the rule of the published 10 m products (notebook 01's MAREA)
DEA_TMP = os.environ.get("MAREA_DEA_TMP")             # DEA's temp root, outside the repository (default below)

In [ ]:
# ── Each site as notebook 01 runs it ────────────────────────────────────
def dutch(site, n_gauges, judges, mouth_side, truth_years, ns):
    d, tag = f"products_{site}_10m", f"eot20_g{n_gauges}"
    return dict(cache=f"ndwi_cube_{site}_2023-2025_10m.nc", extract=f"{d}/marea_extract.npz",
                overpass=f"{d}/overpass_times.json", marea=f"{d}/marea_{tag}",
                nodelay=(f"{d}/comparison_{tag}/products.npz", "z_plain"),
                period=("2023-01-01", "2025-12-31"), n_gauges=n_gauges, judges=judges,
                mouth_side=mouth_side, truth_years=truth_years, ns=ns)


SITES = {
    "escalda": dutch("escalda", 2, ["trnz"], "west", [2025], (60, 50)),
    "wadden":  dutch("wadden", 1, ["harl"], "north+west", [2025, 2024, 2023], (60, 50)),
    "ems":     dutch("ems", 1, ["delf"], "north+west", [2025], (50, 40)),    # 54 screened scenes: 60 is not reachable
}
# ── Validation v3 (experiments/validation_v3_cantabrian_2026-10-29.md): one truth, its cube and production ──
V3_METHODS = ["MAREA", "MAREA_ndwi0", "NIDEM", "DEA", "E4", "E5a", "E5b", "E5c", "E7", "E12",
              "MAREA_B", "MAREA_BL", "MAREA_Ss"]           # the design's methods: the n-tables' ten, MAREA-B, -BL, V15s
if V3KEY:
    from experiments.v3.sites import V3
    from experiments.v3 import windows as v3w
    from experiments.v3 import error_maps as em
    C3 = V3[V3KEY]
    SITE = C3["site"]
    if C3.get("existing_production"):                      # Villaviciosa: its production, as notebook 01 runs it
        prod = dict(extract="products_marea/marea_extract.npz", marea="products_marea/marea_eot20_g0_2023-2025",
                    nodelay=("products_marea/external_products.npz", "no_delay"), record_period=("2023-01-01", "2025-12-31"))
    else:                                                  # notebooks/validation/v3_production_site.ipynb
        ptag = f"{C3['tide_model'].lower()}_g{C3['n_gauges']}"
        prod = dict(extract=f"{C3['out']}/marea_extract.npz", marea=f"{C3['out']}/marea_{ptag}",
                    nodelay=(f"{C3['out']}/comparison_{ptag}/products.npz", "z_plain"), record_period=C3["period"])
    SITES[V3KEY] = dict(cache=C3["cube"], overpass=C3["overpass"], period=C3["period"], n_gauges=C3["n_gauges"],
                        judges=C3["judges"], mouth_side=C3["mouth_side"], truth_years=[int(C3["truth_date"][:4])],
                        ns=C3.get("ns", v3w.NS), **prod)
    EXTRA = "none"                                         # every scene of a v3 window is in its cube
    if C3.get("maps_only"):                                # Santander, Santoña: elevation maps, no truth, no mask
        Q_CLASS = 0.0
    if not os.environ.get("NB_METHODS"):
        METHODS = V3_METHODS
P = SITES[V3KEY or SITE]
TAG = V3KEY or SITE                                        # the outputs' name: the truth's key in v3 (Urdaibai has two)
NS = [int(v) for v in NS.split(",")] if NS else list(P["ns"])
print(f"{SITE}: windows n = {NS} on the rows of {ROW_YEARS}; estimates {METHODS}")
print(f"  cube {P['cache']} · extraction {P['extract']} · boundary EOT20 + {P['n_gauges'] or 0} gauge(s), "
      f"held out {P['judges']}")

## 0 · Setup

As notebook 01: every path is relative to the repository root (opened from `notebooks/validation/`, the first lines
move there); no Copernicus connection is needed; DEA's temp root defaults to `v19_tmp` next to the repository. The
second cell fixes the labels, the roles and the figure style (the muted palette; MAREA in the accent colour).

In [ ]:
import sys
while not os.path.isdir("pyintertidal") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")                                    # from notebooks/validation/ to the repository root
sys.path.insert(0, os.getcwd())

import glob, json, time, types, zlib
from collections import Counter
import numpy as np
import pandas as pd
import xarray as xr
import yaml
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.special import log_ndtr, ndtr, ndtri      # log Φ, Φ and Φ⁻¹ of the standard normal
from scipy.stats import spearmanr

import pyintertidal as pit
from pyintertidal import SentinelCube, marea
from pyintertidal import tide_estimators as te
from pyintertidal import waterline_contour as wc
from pyintertidal.elevation import _fit_block           # the per-pixel fit inside marea.invert_series
from pyintertidal.boundary import make_boundary, nearest_gauges
from experiments.fixedcfg import methods as fm
from experiments.fixedcfg import dea_intertidal as dea
from experiments.fixedcfg.elev_base import NotComputable
from experiments.fixedcfg.level_series import LevelSeries
from experiments import light_to_extraction as lte     # the production rule of the boundary before 2023
from experiments import zarr_cubes as zc               # the band-subset Zarr stores of the pre-2023 years

pit.net.use_system_certificates()
os.makedirs(OUT, exist_ok=True)
DEA_TMP = DEA_TMP or os.path.join(os.path.dirname(os.getcwd()), "v19_tmp")
print("repository", os.getcwd(), "· pyintertidal", pit.__version__, "· DEA temp root", DEA_TMP)

In [ ]:
LABELS = {"MAREA": "V0 MAREA", "MAREA_ndwi0": "V1 MAREA-0", "MAREA_B": "V2 MAREA-B",
          "MAREA_BL": "V10 MAREA-B, lapse rates", "MAREA_B_nodelay": "V2d MAREA-B, no delay",
          "MAREA_nodelay": "MAREA, no delay", "E4": "LogiTide2DEM", "E4_delay": "LogiTide2DEM + MAREA delay",
          "E5c": "sampled exceedance", "DEA": "DEA Intertidal", "NIDEM": "NIDEM", "E5a": "Flood2Topo",
          # the addendum of 2026-10-28 (run when NB_METHODS names them) and the competitors its curves need
          "MAREA_M": "V11 MAREA-M", "MAREA_M_nodelay": "V11d MAREA-M, no delay", "MAREA_P": "V12 MAREA-P",
          "MAREA_H": "V4 MAREA-H (Huber)", "MAREA_H0": "V4-0 MAREA-H, NDWI = 0",
          "MAREA_B_abst": "V13a MAREA-B, abstaining", "MAREA_B_otsu": "V2 MAREA-B + Otsu",
          "MAREA_BL_otsu": "V10 MAREA-BL + Otsu", "MAREA_M_otsu": "V11 MAREA-M + Otsu",
          "E4_otsu": "LogiTide2DEM + Otsu", "E5c_otsu": "sampled exceedance + Otsu", "E12_otsu": "wet/dry bracket + Otsu",
          "E5b": "Tseng linear scaling", "E7": "Coast X-Ray", "E12": "wet/dry bracket",
          "MAREA_S": "V15 MAREA-S (one model, band lags)", "MAREA_Ss": "V15s MAREA-S, smoothed delay"}
SHORT = {"MAREA": "V0", "MAREA_ndwi0": "V1", "MAREA_B": "V2", "MAREA_BL": "V10", "MAREA_B_nodelay": "V2d",
         "MAREA_nodelay": "V0 τ=0", "E4": "LT2D", "E4_delay": "LT2D+τ", "E5c": "exceed.", "DEA": "DEA", "NIDEM": "NIDEM",
         "E5a": "F2T", "MAREA_M": "V11", "MAREA_M_nodelay": "V11d", "MAREA_P": "V12", "MAREA_H": "V4", "MAREA_H0": "V4-0",
         "MAREA_B_abst": "V13a", "MAREA_B_otsu": "V2+O", "MAREA_BL_otsu": "V10+O", "MAREA_M_otsu": "V11+O",
         "E4_otsu": "LT2D+O", "E5c_otsu": "exceed.+O", "E12_otsu": "WDB+O", "E5b": "Tseng", "E7": "CXR", "E12": "WDB",
         "MAREA_S": "V15", "MAREA_Ss": "V15s"}
ROLE = {m: "diagnostic" for m in ("MAREA_ndwi0", "MAREA_B", "MAREA_BL", "MAREA_B_nodelay", "MAREA_nodelay",
                                  "E4_delay", "MAREA_M", "MAREA_M_nodelay", "MAREA_P", "MAREA_H", "MAREA_H0",
                                  "MAREA_B_abst", "MAREA_B_otsu", "MAREA_BL_otsu", "MAREA_M_otsu", "E4_otsu",
                                  "E5c_otsu", "E12_otsu", "MAREA_S", "MAREA_Ss")}
ROLE.update({"MAREA": "reference", **{m: "method" for m in ("E4", "E5c", "DEA", "NIDEM", "E5a", "E5b", "E7", "E12")}})
ORDER = [m for m in LABELS if m in METHODS]
NOT_COMMON = {"MAREA_B_abst"}      # an abstaining estimate never narrows the common pixels of the others

PAL = dict(deep="#2F5D7C", shallow="#8DB3C7", sand="#C9A66B", dark_sand="#8C6D46", seagrass="#5B7F3A",
           grey="#B8B2A3", accent="#B5523B", ink="#2B2B2B", secondary="#6B6B6B")
COLORS = {"MAREA": PAL["accent"], "MAREA_nodelay": PAL["accent"], "MAREA_ndwi0": PAL["sand"],
          "MAREA_B": PAL["dark_sand"], "MAREA_BL": PAL["dark_sand"], "MAREA_B_nodelay": PAL["dark_sand"],
          "E4": PAL["deep"], "E4_delay": PAL["deep"], "E5c": PAL["seagrass"], "DEA": PAL["shallow"], "NIDEM": PAL["grey"],
          "E5a": PAL["secondary"], "MAREA_M": PAL["accent"], "MAREA_M_nodelay": PAL["accent"], "MAREA_P": PAL["accent"],
          "MAREA_H": PAL["sand"], "MAREA_H0": PAL["sand"], "MAREA_B_abst": PAL["dark_sand"],
          "MAREA_B_otsu": PAL["dark_sand"], "MAREA_BL_otsu": PAL["dark_sand"], "MAREA_M_otsu": PAL["accent"],
          "E4_otsu": PAL["deep"], "E5c_otsu": PAL["seagrass"], "E12_otsu": PAL["ink"], "E5b": PAL["grey"],
          "E7": PAL["secondary"], "E12": PAL["ink"], "MAREA_S": PAL["accent"], "MAREA_Ss": PAL["accent"]}
MARKERS = {"MAREA": "o", "MAREA_nodelay": "o", "MAREA_ndwi0": "D", "MAREA_B": "s", "MAREA_BL": "h",
           "MAREA_B_nodelay": "s", "E4": "^", "E4_delay": "^", "E5c": "v", "DEA": "P", "NIDEM": "X", "E5a": "<",
           "MAREA_M": "*", "MAREA_M_nodelay": "*", "MAREA_P": "p", "MAREA_H": "d", "MAREA_H0": "d", "MAREA_B_abst": "s",
           "MAREA_B_otsu": "s", "MAREA_BL_otsu": "h", "MAREA_M_otsu": "*", "E4_otsu": "^", "E5c_otsu": "v",
           "E12_otsu": "x", "E5b": ">", "E7": "8", "E12": "x", "MAREA_S": "H", "MAREA_Ss": "H"}
WSTYLE = [("-", "o", PAL["ink"]), ("--", "s", PAL["secondary"]), (":", "^", PAL["grey"])]   # one per window
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.titleweight": "bold", "axes.labelsize": 8,
                     "axes.edgecolor": PAL["secondary"], "axes.labelcolor": PAL["ink"], "text.color": PAL["ink"],
                     "xtick.color": PAL["secondary"], "ytick.color": PAL["secondary"], "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": PAL["grey"], "grid.alpha": 0.35, "grid.linewidth": 0.5, "legend.frameon": False,
                     "legend.fontsize": 7, "lines.linewidth": 1.3, "lines.markersize": 4.5, "savefig.dpi": 200,
                     "savefig.bbox": "tight", "figure.dpi": 100})

## 1 · The site, the screened scenes, the truth and the windows

Notebook 01's sections 1 to 8, cell for cell: the polygon and the gauges (the N nearest IOC gauges not held out as
judges), the rows (one per survey year), the cached cube, the production screening, the boundary and the screened
list S (checked against the production MAREA record), the optional pre-2023 scenes, the extraction, the mouth distance
and the 3-km bands, the truth and F, the windows, the levels h_B(t − τ) and the 1-min level series of Flood2Topo.
Only the windows of the rows in `ROW_YEARS` are run: at the Wadden the 2024 and 2023 survey groups hold 37 and 20
screened images from 2023–2025, so no window of 50 images reaches them without pre-2023 scenes. F is built from every
row, as in notebook 01, so the pixels are its pixels.

In [ ]:
aoi = pit.sites.get(SITE)
lat_c, lon_c = aoi.centroid
print(aoi, f"· centroid {lat_c:.3f}, {lon_c:.3f}")
near = nearest_gauges(lat_c, lon_c, 6)
members = [g[0] for g in near if g[0] not in P["judges"]][:(P["n_gauges"] or 0)]
for code, name, km in near:
    role = "boundary" if code in members else "judge (held out)" if code in P["judges"] else "unused"
    print(f"  {code:5s} {name:28s} {km:5.1f} km   {role}")

In [ ]:
if V3KEY and not C3.get("existing_production"):        # v3: the production's AOI, the download box
    from pyintertidal.aoi import AOI
    aoi = AOI.from_bbox(**C3["box"], name=f"{SITE} (v3 box)")
    lat_c, lon_c = aoi.centroid
    near = nearest_gauges(lat_c, lon_c, 6)
    members = [g[0] for g in near if g[0] not in P["judges"]][:(P["n_gauges"] or 0)]
    print(aoi, f"· centroid {lat_c:.3f}, {lon_c:.3f}; boundary gauge(s) {members} (expected {C3['expected_gauge']})")
    assert C3["expected_gauge"] in members, "not the production's boundary"

In [ ]:
years = P["truth_years"]
if V3KEY:        # v3: one row, the truth's key, dated at the truth date (its windows are centred there)
    ROWS = [dict(site=V3KEY, year=years[0], end=C3["truth_date"])]
    ROW_YEARS = [years[0]]
elif ONE_ROW:    # one row per site: every cell with its most recent survey of these years, one window to the last
    ROWS = [dict(site=SITE, year=years[0], end=f"{years[0]}-12-31")]
else:
    ROWS = [dict(site=f"{SITE}-{y}" if len(years) > 1 else SITE, year=y, end=f"{y}-12-31") for y in years]
for row in ROWS:
    print(f"row {row['site']:12s} truth dated {row['end']}: its windows end there"
          + ("" if row["year"] in ROW_YEARS else " (not analysed here)"))

In [ ]:
cube = SentinelCube(aoi, P["period"], water="ndwi", cache_path=P["cache"], resolution=10)
cube.ensure(None)                                      # cached: no connection needed
transform, crs = cube.grid
H, W = cube.shape
print(f"{len(cube.dates)} scenes {cube.dates[0]} → {cube.dates[-1]} · grid {H}×{W} @ {abs(transform.a):g} m")

### The screening and the screened list S

The production screening (reference map voted by the clean scenes, then each scene's cloud share over the
transition zone; usable at most 10 % cloudy), the production boundary, S (usable, with an overpass time and a finite
level), and the optional pre-2023 scenes of `experiments/light_to_extraction.py`, put before S. With them, as in
notebook 01, `boundary` becomes a `SplitBoundary`: before 2023-01-01 the production rule of those years,
`pre2023_boundary` (production's object with its gauge members extended by the IOC records of those years, demeaned
by production's constants), from then on production's object, chosen instant by instant; every level is read from it.

In [ ]:
t0 = time.perf_counter()
reference_map, cloud_pct = pit.reference_and_clouds(
    cube, threshold=0.0, bad_classes=pit.water.BAD_CLASSES, clean_scene_max_bad=0.05,
    stable_threshold=0.95, transition_buffer_px=10)
usable = pit.filter_dates(cloud_pct, cloud_threshold=0.10)
for code, name in [(0, "transition"), (1, "stable water"), (2, "stable land")]:
    print(f"  class {code} ({name:13s}): {int((reference_map == code).sum()):>10,} px")
print(f"{len(usable)} of {len(cloud_pct)} scenes usable  [{time.perf_counter() - t0:.0f} s]")

In [ ]:
boundary = make_boundary(aoi, tide_model="EOT20", n_gauges=P["n_gauges"], period=P["period"],
                         exclude=P["judges"])
print(f"boundary: {boundary.name}")
overpass = {d: pd.Timestamp(v) for d, v in json.load(open(P["overpass"])).items()}
cand = [d for d in usable if d in overpass]
t_cand = pd.DatetimeIndex([overpass[d] for d in cand]).tz_localize(None)
ok = np.isfinite(np.asarray(boundary.levels(t_cand), float))
S_dates, S_t = np.array(cand)[ok], t_cand[ok]
print(f"S: {len(S_dates)} screened scenes {S_dates[0]} → {S_dates[-1]} ({len(usable) - len(cand)} usable "
      f"scenes without an overpass time, {int((~ok).sum())} without a finite level)")

record = [d[:10] for d in json.load(open(f"{P['marea']}/result.json"))["record_dates"]]
part = [d for d in S_dates if "2023-01-01" <= d <= "2025-12-31"]
print(f"S in 2023-2025 is the production MAREA record ({len(record)} scenes): {part == record}")

In [ ]:
if V3KEY:        # v3: S over the production record's period is that record (the same screening and boundary)
    r0, r1 = P["record_period"]
    part = [d for d in S_dates if r0 <= d <= r1]
    print(f"v3: S in {r0} → {r1} is the production MAREA record of {P['marea']} ({len(record)} scenes): "
          f"{part == record}")

In [ ]:
class SplitBoundary:
    """A boundary that reads `before` at the instants before `cut` and `after` from `cut` on, instant by instant.
    Each is called once, on its own instants in the order given (sorted in, sorted out, as production's object
    needs)."""

    def __init__(self, before, after, cut):
        self.before, self.after, self.cut = before, after, pd.Timestamp(cut)
        self.name = f"{after.name} (before {self.cut.date()}: {before.name})"

    def levels(self, times):
        t = pd.DatetimeIndex(times)
        if t.tz is not None:
            t = t.tz_localize(None)
        early = np.asarray(t < self.cut)
        h = np.full(len(t), np.nan)
        if (~early).any():
            h[~early] = np.asarray(self.after.levels(t[~early]), float)
        if early.any():
            h[early] = np.asarray(self.before.levels(t[early]), float)
        return h

In [ ]:
found = sorted(glob.glob(f"results/fixedcfg/extra/{SITE}_extract_*.npz"))
EXTRA_PATH = (found[-1] if found else None) if EXTRA == "auto" else (None if EXTRA == "none" else EXTRA)
x_dates = np.array([], str)
boundary_prod = getattr(boundary, "after", boundary)   # production's boundary object (also when the cell runs again)
if EXTRA_PATH:
    side = json.load(open(EXTRA_PATH[:-4] + ".json", encoding="utf-8"))
    boundary_pre, pre_info = lte.pre2023_boundary(SITE, boundary_prod, lte.PRODUCTION_GAUGE_CACHE,
                                                  lte.PRE2023_GAUGE_CACHE)
    boundary = SplitBoundary(boundary_pre, boundary_prod, P["period"][0])
    print(f"boundary before {P['period'][0]}: {boundary_pre.name}; gauge weights {pre_info['weights']}")
    cloud_x = {d: p for d, p in side["cloud_pct"].items() if p is not None and d < P["period"][0]}
    cand_x = [d for d in pit.filter_dates(cloud_x, cloud_threshold=0.10) if side["overpass"].get(d)]
    t_x = pd.DatetimeIndex([pd.Timestamp(side["overpass"][d]) for d in cand_x]).tz_localize(None)
    ok_x = np.isfinite(np.asarray(boundary.levels(t_x), float))
    x_dates = np.array(cand_x)[ok_x]
    S_dates, S_t = np.concatenate([x_dates, S_dates]), t_x[ok_x].append(S_t)
    print(f"{EXTRA_PATH}: {len(x_dates)} screened scenes before {P['period'][0]} put before S "
          f"-> {len(S_dates)} scenes {S_dates[0]} → {S_dates[-1]}")
else:
    print("no extra extraction: the windows hold the cube's scenes only; every level is production's boundary")

### The extraction, the mouth distance and the bands

`marea_extract.npz`: the production pixels, the NDWI of every scene (NaN where undefined) and the clear mask
C = SCL in {4, 5, 6, 7}; only the rows of S stay in memory. The mouth distance s and MAREA's 3-km bands, as
`marea.reconstruct` computes them.

In [ ]:
ex = np.load(P["extract"])
keep = ex["keep"]                                      # flat indices of the pixels in the H x W frame
pos = {d: i for i, d in enumerate(ex["dates"].astype(str))}
print(f"{len(keep):,} pixels · {len(pos)} scenes · the cube's dates: {list(pos) == list(cube.dates)}")
main = [pos[d] for d in S_dates[len(x_dates):]]
Y_S, C_S = ex["Y"][main], ex["C"][main]                # (scenes of S, pixels): NDWI and clear
if len(x_dates):
    X = np.load(EXTRA_PATH)
    assert np.array_equal(X["keep"], keep), "the extra extraction is on other pixels"
    xpos = {d: i for i, d in enumerate(X["dates"].astype(str))}
    rows_x = [xpos[d] for d in x_dates]
    Y_S, C_S = np.concatenate([X["Y"][rows_x], Y_S]), np.concatenate([X["C"][rows_x], C_S])
print(f"Y and C of S: {Y_S.shape[0]} scenes × {Y_S.shape[1]:,} pixels")

In [ ]:
seeds = pit.geometry.mouth_seeds(ex["sea"], side=P["mouth_side"])
s_km = pit.geometry.along_distance(ex["sea"] | ex["inter"], seeds, 10).ravel()[keep] / 1000.0
BAND_KM = 3.0
_, centers, band_of = te.make_bands_fixed(s_km, BAND_KM, 500)
production = np.load(f"{P['marea']}/marea.npz")
print(f"{len(centers)} bands of {BAND_KM:g} km, centres {np.round(centers, 1)} km; "
      f"the production product's bands: {np.array_equal(band_of, production['band'])}")

### The truth and the pixels F

Every pixel reads the 20 m Vaklodingen cell under it (`sample_truth`), year by year from 2025 down to 2023: the most
recent survey with a value gives its truth and its survey year. The truth is kept inside the production no-delay
range ± 0.25 m; F is those pixels in the survey years of the rows; above 200,000 pixels, v19's seeded subsample.

In [ ]:
import pyproj
from experiments.v0_vaklodingen_fetch import sheets_for
from experiments.v1_vaklodingen_validate import sample_truth


def vaklodingen(year):
    """{sheet: (x, y, z, year)} of the cells surveyed in `year` (a later survey of that year wins)."""
    cells = {}
    for name, _, _ in sheets_for(SITE):
        path = f"data_v4/truth/vaklodingen/{name}"
        if not os.path.exists(path):
            continue
        with xr.open_dataset(path) as ds:
            k = np.flatnonzero(ds["time"].dt.year.values == year)
            if not len(k):
                continue
            z, x, y = ds["z"].isel(time=k).values, ds["x"].values, ds["y"].values
        zt = np.full(z.shape[1:], np.nan)
        for zk in z:
            zt[np.isfinite(zk)] = zk[np.isfinite(zk)]
        cells[name] = (x, y, zt, np.full(zt.shape, year))
    return cells

In [ ]:
if V3KEY and C3.get("maps_only"):   # maps only: no truth; F is set below from the production's mapped pixels
    truth = np.full(len(keep), np.nan)
    group = np.full(len(keep), ROWS[0]["year"])
    survey, TRUTH_META = group.copy(), None
    print(f"{V3KEY}: maps only, no truth")
elif V3KEY:      # v3: the PNOA DTM truth on the cube's grid (experiments/v3/pnoa_dtm_truth.py), read as it is
    import rasterio
    with rasterio.open(C3["truth"]) as r:
        assert (r.height, r.width) == (H, W), f"{C3['truth']} is {r.height} x {r.width}, the cube {H} x {W}"
        assert np.allclose(tuple(r.transform)[:6], tuple(transform)[:6]) and r.crs == crs, "not on the cube's grid"
        z_truth = r.read(1).astype(float).ravel()
    TRUTH_META = json.load(open(os.path.splitext(C3["truth"])[0] + ".json", encoding="utf-8"))
    truth = z_truth[keep]
    group = np.where(np.isfinite(truth), ROWS[0]["year"], 0)
    survey = group.copy()
    print(f"{C3['truth']}: truth date {TRUTH_META['truth_date']} ({TRUTH_META['truth_date_rule']}); "
          f"{np.isfinite(truth).sum():,} of {len(keep):,} production pixels with a value; water patches "
          f"{TRUTH_META['water_patch_share']:.1%} of the sheets' cells")
else:
    with xr.open_dataset(P["cache"]) as ds:
        xs, ys = ds["x"].values, ds["y"].values
    to_rd = pyproj.Transformer.from_crs(crs, "EPSG:28992", always_xy=True)
    xq, yq = to_rd.transform(xs[keep % W], ys[keep // W])
    truth, group = np.full(len(keep), np.nan), np.zeros(len(keep), int)
    for year in (2025, 2024, 2023):                    # the most recent survey with a value wins
        v, _ = sample_truth(vaklodingen(year), xq, yq)
        new = np.isfinite(v) & (group == 0)
        truth[new], group[new] = v[new], year
    survey = group.copy()                              # the year of each cell's survey
    if ONE_ROW:
        group[np.isin(group, P["truth_years"])] = ROWS[0]["year"]   # the site's truth years in its one row; a
                                                       # cell surveyed in another year stays out, as per year

In [ ]:
path, key = P["nodelay"]
z_nodelay = np.load(path)[key]
lo, hi = np.nanmin(z_nodelay) - RANGE_PAD_M, np.nanmax(z_nodelay) + RANGE_PAD_M
in_range = (truth >= lo) & (truth <= hi)
row_years = [row["year"] for row in ROWS]
for g in np.unique(group[np.isfinite(truth)]):
    print(f"  survey {g or 'LiDAR'}: {int((group == g).sum()):,} px with a value, "
          f"{int(((group == g) & in_range).sum()):,} inside [{lo:+.2f}, {hi:+.2f}] m"
          f" ({'a row' if g in row_years else 'left out'})")

F = np.flatnonzero(in_range & np.isin(group, row_years))
if len(F) > MAX_PX:                                    # v19's seeded subsample: the same pixels as v19
    rng = np.random.default_rng(np.random.SeedSequence([SEED, zlib.crc32(SITE.encode()) & 0x7FFFFFFF, 7]))
    F = F[np.sort(rng.permutation(len(F))[:MAX_PX])]
for row in ROWS:
    row["px"] = F[group[F] == row["year"]]
    print(f"row {row['site']:12s}: F = {len(row['px']):,} pixels")

In [ ]:
if V3KEY and C3.get("maps_only"):   # maps only: every pixel the production's no-delay inversion maps
    F = np.flatnonzero(np.isfinite(z_nodelay))
    ROWS[0]["px"] = F
    print(f"{V3KEY}: maps on the {len(F):,} pixels of the production's no-delay product")

### The windows and the water levels

Window(n) is the n most recent images of S on or before the row's truth date. The levels h_B(t − τ) are read once for
every scene of S and every whole minute τ from −45 to 180 (MAREA's lag bank and its per-pixel delays; τ = 0 is what
every method without a delay reads); `make_obs` gives a method its inputs. Flood2Topo also gets the 1-min level series.

In [ ]:
windows = []
for row in ROWS:
    if row["year"] not in ROW_YEARS:
        continue
    if V3KEY:    # v3: the n screened scenes nearest the truth date, a window symmetric in time inside the cube;
                 # one the cube cannot hold about d is centred at the nearest date that lets it (amendment 2026-11-03)
        cover = (cube.dates[0], cube.dates[-1])
        for n in NS:
            sw = v3w.fitting_window(S_dates, row["end"], n, cover)
            win = dict(site=row["site"], year=row["year"], end=row["end"], truth_date=row["end"], n=n,
                       px=row["px"], available=len(S_dates), scenes=sw["idx"] if sw["reachable"] else None,
                       half_width_days=sw["half_width_days"], n_before=sw["n_before"], n_after=sw["n_after"],
                       centre=sw["centre"], shift_days=sw["shift_days"])
            if win["scenes"] is None:
                win["why"] = (f"{len(S_dates)} screened scenes: no window of {n} holding the truth date "
                              f"{row['end']} fits in the cube's {cover[0]} → {cover[1]}")
                print(f"{row['site']:14s} n={n:3d}: not reachable, {win['why']}")
            else:
                d = S_dates[win["scenes"]]
                win["end"] = str(d[-1])                # the window's last scene (the level series reach it)
                per_year = dict(sorted(Counter(str(x)[:4] for x in d).items()))
                moved = f", centred {sw['centre']} ({sw['shift_days']:+d} d)" if sw["shift_days"] else ""
                print(f"{row['site']:14s} n={n:3d}: {d[0]} → {d[-1]}  T = {sw['half_width_days']:.0f} d{moved}, "
                      f"{sw['n_before']} before / {sw['n_after']} after the truth date; per year {per_year}")
            windows.append(win)
        continue
    upto = np.flatnonzero(S_dates <= row["end"])
    for n in NS:
        win = dict(site=row["site"], year=row["year"], end=row["end"], n=n, px=row["px"], available=len(upto),
                   scenes=upto[-n:] if len(upto) >= n else None)
        windows.append(win)
        if win["scenes"] is None:
            print(f"{row['site']:12s} n={n:3d}: not reachable, {len(upto)} screened images on or before {row['end']}")
        else:
            d = S_dates[win["scenes"]]
            per_year = dict(sorted(Counter(str(x)[:4] for x in d).items()))
            print(f"{row['site']:12s} n={n:3d}: {d[0]} → {d[-1]}  images per year {per_year}")

In [ ]:
if V3KEY:        # v3: the class-dominance mask of the design, q <= f_wet <= 1 - q over the window's clear looks
    for win in windows:
        if win["scenes"] is None:
            continue
        masks = v3w.class_mask_sensitivity(Y_S[:, win["px"]], C_S[:, win["px"]], Q_SENS, rows=win["scenes"])
        q0 = masks[0.0]                                # every pixel of F with a clear look in the window
        win["px_F"], win["px"] = win["px"], win["px"][q0]
        win["qmask"] = {q: m[q0] for q, m in masks.items()}   # on the estimated pixels win["px"]
        print(f"{win['site']} n={win['n']}: F {len(win['px_F']):,} px, with a clear look {len(win['px']):,}; "
              + ", ".join(f"q={q:g}: {int(m.sum()):,}" for q, m in win["qmask"].items()))

In [ ]:
shifted = {tau: S_t - pd.Timedelta(minutes=tau) for tau in range(-45, 181)}
inst = pd.DatetimeIndex(np.unique(np.concatenate([s.values for s in shifted.values()])))
lev = np.asarray(boundary.levels(inst), float)
LEV = {tau: lev[np.searchsorted(inst.values, s.values)] for tau, s in shifted.items()}
h_direct = np.asarray(boundary.levels(S_t), float)
print(f"h_B(t - tau) at {len(S_t)} scenes, tau = -45..180 min; at tau = 0 it is the level read directly "
      f"(max |diff| {np.max(np.abs(LEV[0] - h_direct)):.1e} m); range [{LEV[0].min():+.2f}, {LEV[0].max():+.2f}] m")


def level_at(tau):
    """h_B(t - tau) at every scene of S; a delay outside the table is read from the boundary itself."""
    if tau not in LEV:
        LEV[tau] = np.asarray(boundary.levels(S_t - pd.Timedelta(minutes=tau)), float)
    return LEV[tau]


def make_obs(win, px, tau=0):
    """The common inputs of a window at the pixels px (positions in keep), with the levels read tau minutes
    earlier (0 for every method; the '+ MAREA delay' variants pass MAREA's delay)."""
    w = win["scenes"]
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    with np.errstate(invalid="ignore"):
        wet = C & (Y > 0)                              # an undefined NDWI is never wet
    return types.SimpleNamespace(Y=Y, C=C, W=wet, h=level_at(tau)[w], t=S_t[w], keep=keep[px],
                                 shape=(H, W), pixel_m=10.0, site=SITE)

In [ ]:
LS = None
reach = [w for w in windows if w["scenes"] is not None]
if reach and any("level_series" in fm.METHODS[m][2].get("needs", []) for m in METHODS if m in fm.METHODS):
    t0 = time.perf_counter()
    first = min(S_t[w["scenes"][0]] for w in reach).normalize()
    last = pd.Timestamp(max(w["end"] for w in reach)) + pd.Timedelta(days=1)
    LS = LevelSeries.from_boundary(boundary, first, last)
    print(f"1-min level series {first.date()} → {last.date()}: {len(LS.h):,} levels  [{time.perf_counter() - t0:.0f} s]")

### The water level before 2023: the checks

With pre-2023 scenes: (1) the levels of the 2023–2025 scenes, at every delay of the table, are those production's
boundary gives in a run without pre-2023 scenes (one call over the same instants), bit for bit; (2) the levels of
the pre-2023 scenes without delay are the extraction's record (`levels_pre2023`, written by
`light_to_extraction.py build` with the same `pre2023_boundary`), bit for bit, and the record's boundary, tide
member, gauge weights and demeaning constants are those built here; (3) the 1-min level series switches from
`pre2023_boundary` to production's boundary at 2023-01-01 (the two days around it, bit for bit).

In [ ]:
if len(x_dates):
    cut = boundary.cut
    post = np.flatnonzero(S_t >= cut)                  # the 2023-2025 scenes of S
    sh = {tau: S_t[post] - pd.Timedelta(minutes=tau) for tau in range(-45, 181)}
    inst_p = pd.DatetimeIndex(np.unique(np.concatenate([s.values for s in sh.values()])))
    lev_p = np.asarray(boundary_prod.levels(inst_p), float)          # as a run without pre-2023 scenes reads them
    assert all(np.array_equal(LEV[tau][post], lev_p[np.searchsorted(inst_p.values, s.values)])
               for tau, s in sh.items()), "the 2023-2025 levels are not production's"
    print(f"2023-2025: the levels of its {len(post)} scenes at tau = -45..180 min are production's, bit for bit")

    rec = side["levels_pre2023"]
    for k in ("boundary", "production_boundary", "tide_member", "weights"):
        assert rec[k] == pre_info[k], f"the extraction's {k} is {rec[k]!r}, here {pre_info[k]!r}"
    for g, v in pre_info["gauges"].items():
        for k in ("production_median_m", "pre2023_median_m", "samples"):
            assert rec["gauges"][g][k] == v[k], f"gauge {g}: the extraction's {k} is {rec['gauges'][g][k]}, here {v[k]}"
    have = [i for i, d in enumerate(x_dates) if (rec["per_date"].get(d) or {}).get("h_B") is not None]
    h_rec = np.array([rec["per_date"][x_dates[i]]["h_B"] for i in have], float)
    assert np.array_equal(LEV[0][have], h_rec), \
        f"the pre-2023 levels are not the extraction's record (max |diff| {np.max(np.abs(LEV[0][have] - h_rec)):.1e} m)"
    kinds = Counter(rec["per_date"][x_dates[i]]["members"] for i in have)
    print(f"before 2023: the levels of {len(have)} of {len(x_dates)} scenes are the extraction's record "
          f"(levels_pre2023), bit for bit, with its boundary, tide member, gauge weights and demeaning constants; "
          f"members at those scenes {dict(kinds)}")

    if LS is not None and LS.t[0] < cut:
        a, b = LS.t.searchsorted(cut - pd.Timedelta(days=1)), LS.t.searchsorted(cut + pd.Timedelta(days=1))
        t = LS.t[a:b]
        h = np.concatenate([boundary_pre.levels(t[t < cut]), boundary_prod.levels(t[t >= cut])])
        assert np.array_equal(LS.h[a:b], h), "the 1-min series does not switch boundary at the cut"
        print(f"1-min series: its {b - a:,} levels around {cut.date()} are pre2023_boundary's before it and "
              "production's from it, bit for bit")
else:
    print("no pre-2023 scene: every level is production's boundary")

## 2 · The estimates, per pixel

**V0 MAREA**, as notebook 01 runs it: the lag of every 3-km band refitted on the window's images by the profiled
Bernoulli likelihood (`te.m2a_rasch`, production's settings from `configs/m2.yaml`, its seeded draw), a negative band
lag set to 0, the profile smoothed along the mouth distance and rounded to the minute (τ_px); every pixel inverted
with the level read τ_px minutes earlier (`marea.invert_series`: the least-squares fit NDWI = a + bΦ((h − z)/σ) on a
grid, elevation = the midpoint z), between the window's own lowest and highest level. `marea_delays` is notebook
01's function; here it runs once per window (`window_delays`, the same seeded fit) and every delayed estimate reuses
it, so its seconds are a row of their own (`delay_fit`).

In [ ]:
M2 = yaml.safe_load(open("configs/m2.yaml", encoding="utf-8"))
FIT_KW = dict(sigma0=M2["sigma0_m"], sg_grid=tuple(M2["sigma_perfil_m"]), tau_bounds=marea.TAU_BOUNDS,
              z_points=M2["z_puntos"], max_px_band=M2["max_px_banda"]["m2a"], monotone=True)


def marea_delays(win):
    """MAREA's band lags on the window's images, and the delay of every pixel of keep to the minute."""
    w = win["scenes"]
    bank = te.ShiftBank(marea.BANK_TAUS, [LEV[int(tau)][w] for tau in marea.BANK_TAUS])
    with np.errstate(invalid="ignore"):
        wet = C_S[w] & (Y_S[w] > 0)
    fit = te.m2a_rasch(wet, C_S[w], bank, band_of, centers, len(centers),
                       rng=np.random.default_rng(M2["seed"] + 31), **FIT_KW)
    tau_bands = np.clip(np.asarray(fit["tau"], float), 0, None)       # causality: a negative band lag is 0
    tau_px = te.smooth_lag_profile(centers, tau_bands, s_km, bandwidth_km=BAND_KM)
    return tau_bands, np.where(np.isfinite(tau_px), np.round(tau_px), 0.0)


def lags_text(tau_bands):
    return "band lags " + "/".join(f"{v:.0f}" for v in tau_bands) + " min"


DELAYS = {}


def window_delays(win):
    """marea_delays of the window, fitted once (the fit is seeded: every call would give the same delays)."""
    key = (win["site"], win["n"])
    if key not in DELAYS:
        DELAYS[key] = marea_delays(win)
    return DELAYS[key]


def run_marea(win, px):
    """V0: notebook 01's run_marea (its value and its note), with the window's delays fitted once."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win)
    tau_px = tau_px[px]
    h0 = LEV[0][w]
    Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
    z, sigma = np.full(len(px), np.nan), np.full(len(px), np.nan)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], sigma[cols] = marea.invert_series(Y0[:, cols], C0[:, cols], LEV[int(tau)][w], h0.min(), h0.max(),
                                                   min_bracket=MAREA_MIN_BRACKET)
    return z, lags_text(tau_bands), None, {"sigma": sigma}

**V1 MAREA-0** (diagnostic): V0's fit, read where the fitted curve crosses NDWI = 0: h0 = z + σΦ⁻¹(−a/b), no value
when 0 is not strictly between a and a + b. `marea_fit` is notebook 01's `invert_ndwi0` keeping the whole fit (a, b,
σ before the guards, and the midpoint), so that V0's a, b and σ are saved per pixel; every window asserts that its
midpoint is V0's z bit for bit.

In [ ]:
def marea_fit(Y, C, h, lo, hi, min_bracket=0):
    """Notebook 01's invert_ndwi0, keeping the fit: dict(z0, z_mid, a, b, sigma). The fit is marea.invert_series'
    (_fit_block with its z grid, sigma grid, blocks of 4,000 pixels and guards): z_mid is its midpoint (NaN where
    the guards reject), z0 the level where the fitted NDWI = a + b Phi((h - z)/sigma) crosses 0 (NaN when 0 is not
    strictly between a and a + b); a, b and sigma are the fit's, before the guards."""
    out = {k: np.full(Y.shape[1], np.nan) for k in ("z0", "z_mid", "a", "b", "sigma")}
    grid = np.linspace(lo, hi, 50)
    for j in range(0, Y.shape[1], 4000):
        s = slice(j, min(j + 4000, Y.shape[1]))
        a, b, mu, sg, _, N = _fit_block(np.asarray(Y[:, s], np.float64), np.asarray(C[:, s], np.float64),
                                        h, grid, marea.SG_GRID)
        ok = (N >= 8) & (b > 0.15) & (b < 2.5) & (np.abs(a) < 2.5)          # invert_series' guards
        if min_bracket:
            clear, hh = np.asarray(C[:, s]) > 0, np.asarray(h, float)[:, None]
            below, above = (clear & (hh < mu)).sum(axis=0), (clear & (hh > mu)).sum(axis=0)
            ok &= (below >= min_bracket) & (above >= min_bracket)
        cross = ok & (a < 0) & (a + b > 0)
        out["z_mid"][s] = np.where(ok, mu, np.nan)
        out["z0"][s] = np.where(cross, mu + sg * ndtri(np.where(cross, -a / np.where(cross, b, 1.0), 0.5)), np.nan)
        out["a"][s], out["b"][s], out["sigma"][s] = a, b, sg
    return out


def run_marea_ndwi0(win, px):
    """V1: notebook 01's run_marea_ndwi0 (its value and its note); the fit's a, b, sigma and midpoint as extras."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win)
    tau_px = tau_px[px]
    h0 = LEV[0][w]
    Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
    fit = {k: np.full(len(px), np.nan) for k in ("z0", "z_mid", "a", "b", "sigma")}
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        part = marea_fit(Y0[:, cols], C0[:, cols], level_at(int(tau))[w], h0.min(), h0.max(),
                         min_bracket=MAREA_MIN_BRACKET)
        for k, v in part.items():
            fit[k][cols] = v
    fitted, crossed = int(np.isfinite(fit["z_mid"]).sum()), int(np.isfinite(fit["z0"]).sum())
    note = f"{lags_text(tau_bands)}; {fitted - crossed:,} of {fitted:,} fitted px do not cross NDWI = 0"
    return fit.pop("z0"), note, None, fit

**V2 MAREA-B** (diagnostic, new): V0's delay; per pixel, the probit likelihood of its wet/dry record on its clear
looks, ℓ(z, σ) = Σ_wet log Φ((h − z)/σ) + Σ_dry log Φ(−(h − z)/σ) with h = h_B(t − τ_px) and wet = clear and
NDWI > 0 (a clear look with an undefined NDWI is dry, as in MAREA's lag fit), maximised on `invert_series`' grid: 50
levels between the window's lowest and highest level, σ in `marea.SG_GRID`. The elevation is z, where P(wet) = 0.5.
Under complete separation ℓ has a plateau instead of a maximum: the z levels whose profile log-likelihood (the
maximum over σ) lies within `B_TIE_TOL` of the largest are tied, and z is their median (`B_TIE_TOL` is this notebook's
reading of *tied*). At least 8 clear looks, with at least one wet and one dry. **V2d** is V2 with every delay 0.
`invert_bernoulli` works on blocks of 4,000 pixels, as `_fit_block`; the next cell checks it against plain loops.

In [ ]:
def invert_bernoulli(Y, C, h, lo, hi, min_looks=B_MIN_LOOKS, tie_tol=B_TIE_TOL, mu_points=50, chunk=4000):
    """MAREA-B: per pixel, the probit likelihood of the wet/dry record on the clear looks against the levels h,
    P(wet) = Phi((h - z)/sigma), maximised on invert_series' grid (mu_points levels in [lo, hi], sigma in
    marea.SG_GRID). z = the median of the tied z levels (profile log-likelihood within tie_tol of its maximum);
    sigma = the best sigma at the tied level nearest z. NaN with fewer than min_looks clear looks or without a wet
    and a dry one. Returns (z, sigma, the number of tied levels)."""
    grid, sgs = np.linspace(lo, hi, mu_points), np.asarray(marea.SG_GRID, float)
    u = (np.asarray(h, float)[:, None, None] - grid[None, :, None]) / sgs[None, None, :]
    lw, ld = log_ndtr(u).reshape(len(h), -1), log_ndtr(-u).reshape(len(h), -1)   # (looks, levels x sigmas)
    P = Y.shape[1]
    z, sg, n_tied = np.full(P, np.nan), np.full(P, np.nan), np.zeros(P, int)
    for j in range(0, P, chunk):
        s = slice(j, min(j + chunk, P))
        clear = np.asarray(C[:, s], bool)
        with np.errstate(invalid="ignore"):
            wet = clear & (np.asarray(Y[:, s]) > 0)     # an undefined NDWI is never wet
        dry = clear & ~wet
        ll = (wet.T.astype(float) @ lw + dry.T.astype(float) @ ld).reshape(-1, len(grid), len(sgs))
        prof = ll.max(axis=2)                           # the profile over sigma: (pixels, levels)
        tied = prof >= prof.max(axis=1, keepdims=True) - tie_tol
        zb = np.nanmedian(np.where(tied, grid[None, :], np.nan), axis=1)
        near = np.argmin(np.where(tied, np.abs(grid[None, :] - zb[:, None]), np.inf), axis=1)
        sb = sgs[ll[np.arange(len(near)), near].argmax(axis=1)]
        ok = (clear.sum(axis=0) >= min_looks) & wet.any(axis=0) & dry.any(axis=0)
        z[s], sg[s], n_tied[s] = np.where(ok, zb, np.nan), np.where(ok, sb, np.nan), np.where(ok, tied.sum(1), 0)
    return z, sg, n_tied


def run_marea_b(win, px, delay=True):
    """V2 (delay=True, V0's delays) and V2d (every delay 0), between the window's lowest and highest level."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win) if delay else (None, None)
    tau_px = tau_px[px] if delay else np.zeros(len(px))
    h0 = LEV[0][w]
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    z, sigma, tied = np.full(len(px), np.nan), np.full(len(px), np.nan), np.zeros(len(px), int)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], sigma[cols], tied[cols] = invert_bernoulli(Y[:, cols], C[:, cols], level_at(int(tau))[w],
                                                            h0.min(), h0.max())
    fin = np.isfinite(z)
    note = (f"{lags_text(tau_bands)}; " if delay else "every delay 0; ") + \
        f"{int((tied[fin] > 1).sum()):,} of {int(fin.sum()):,} px on a tied plateau"
    return z, note, None, {"sigma": sigma, "n_tied": tied}

In [ ]:
def brute_force_b(wet, clear, h, lo, hi, tie_tol=B_TIE_TOL):
    """MAREA-B of one pixel by plain loops over the grid (the check of invert_bernoulli)."""
    wet = wet & clear
    if clear.sum() < B_MIN_LOOKS or not wet.any() or not (clear & ~wet).any():
        return np.nan
    grid, prof = np.linspace(lo, hi, 50), []
    for zk in grid:
        best = -np.inf
        for sk in marea.SG_GRID:
            ll = 0.0
            for k in range(len(h)):
                if clear[k]:
                    u = (h[k] - zk) / sk
                    ll += float(log_ndtr(u if wet[k] else -u))
            best = max(best, ll)
        prof.append(best)
    prof = np.array(prof)
    return float(np.median(grid[prof >= prof.max() - tie_tol]))


rng = np.random.default_rng(5)
hc = rng.uniform(-2.0, 2.0, 60)                        # 60 looks, as in the longest window
Yc = np.stack([np.where(rng.random(60) < ndtr((hc - 0.3) / 0.25), 0.4, -0.3),   # a noisy pixel, z = 0.3
               np.where(hc > -0.4, 0.5, -0.2),                                  # complete separation at -0.4
               np.where(hc > 0.8, 0.5, np.nan),                                 # ... at 0.8, dry = undefined NDWI
               np.where(hc > 0.0, 0.5, -0.2)], axis=1)                          # 5 clear looks: no value
Cc = rng.random((60, 4)) > 0.1
Cc[:, 3] = np.arange(60) < 5
zv, sv, nt = invert_bernoulli(Yc, Cc, hc, hc.min(), hc.max())
with np.errstate(invalid="ignore"):
    zb = [brute_force_b(Yc[:, j] > 0, Cc[:, j], hc, hc.min(), hc.max()) for j in range(4)]
assert np.allclose(zv, zb, rtol=0, atol=1e-12, equal_nan=True) and np.isnan(zv[3]), (zv, zb)
print(f"invert_bernoulli = the brute-force maximum on 4 synthetic pixels: z = {np.round(zv, 3)} "
      f"(tied levels {nt.tolist()})")

**V10 MAREA-B with lapse rates** (diagnostic, declared on 2026-10-27 before any confirmatory window): V2 with a
contamination term, P(wet) = e + (1 − 2e)Φ((h − z)/σ), so a few spurious wet looks at low tide or dry looks at high
tide (a missed cloud, a wet patch left on the flat) do not drag z. The lapse rate e is maximised with z and σ on the
grid `B_LAPSES` = {0, 0.02, 0.05, 0.10}; the rest is V2's: V0's delay, `invert_series`' z and σ grids between the
window's lowest and highest level, the median of the tied z levels (the profile now over σ and e), at least 8 clear
looks with a wet and a dry one. With e = 0 the log-likelihood is V2's (log Φ), so `invert_bernoulli_lapse` with
e = 0 alone is V2 bit for bit; the next cell checks that, and checks it against plain loops.

In [ ]:
def invert_bernoulli_lapse(Y, C, h, lo, hi, lapses=B_LAPSES, min_looks=B_MIN_LOOKS, tie_tol=B_TIE_TOL, mu_points=50,
                           chunk=4000):
    """V10: per pixel, the likelihood of its wet/dry record on the clear looks with P(wet) = e + (1 - 2e)
    Phi((h - z)/sigma), maximised on invert_series' grid (mu_points levels in [lo, hi], sigma in marea.SG_GRID) and
    over the lapse rates e in `lapses` (e = 0: log Phi, invert_bernoulli's own terms). z = the median of the tied z
    levels (profile over sigma and e within tie_tol of its maximum); sigma and e = the best pair at the tied level
    nearest z. NaN with fewer than min_looks clear looks or without a wet and a dry one. Returns (z, sigma, e, the
    number of tied levels)."""
    grid, sgs, es = np.linspace(lo, hi, mu_points), np.asarray(marea.SG_GRID, float), np.asarray(lapses, float)
    u = (np.asarray(h, float)[:, None, None] - grid[None, :, None]) / sgs[None, None, :]

    def log_p(v):                                     # log(e + (1 - 2e) Phi(v)) for every e: (..., lapses)
        return np.stack([log_ndtr(v) if e == 0 else np.log(e + (1 - 2 * e) * ndtr(v)) for e in es], axis=-1)

    lw, ld = log_p(u).reshape(len(h), -1), log_p(-u).reshape(len(h), -1)   # (looks, levels x sigmas x lapses)
    P, K = Y.shape[1], len(sgs) * len(es)
    z, sg, lapse, n_tied = np.full(P, np.nan), np.full(P, np.nan), np.full(P, np.nan), np.zeros(P, int)
    for j in range(0, P, chunk):
        s = slice(j, min(j + chunk, P))
        clear = np.asarray(C[:, s], bool)
        with np.errstate(invalid="ignore"):
            wet = clear & (np.asarray(Y[:, s]) > 0)     # an undefined NDWI is never wet
        dry = clear & ~wet
        ll = (wet.T.astype(float) @ lw + dry.T.astype(float) @ ld).reshape(-1, len(grid), K)
        prof = ll.max(axis=2)                           # the profile over sigma and e: (pixels, levels)
        tied = prof >= prof.max(axis=1, keepdims=True) - tie_tol
        zb = np.nanmedian(np.where(tied, grid[None, :], np.nan), axis=1)
        near = np.argmin(np.where(tied, np.abs(grid[None, :] - zb[:, None]), np.inf), axis=1)
        best = ll[np.arange(len(near)), near].argmax(axis=1)   # sigma index * len(es) + e index
        ok = (clear.sum(axis=0) >= min_looks) & wet.any(axis=0) & dry.any(axis=0)
        z[s], n_tied[s] = np.where(ok, zb, np.nan), np.where(ok, tied.sum(1), 0)
        sg[s] = np.where(ok, sgs[best // len(es)], np.nan)
        lapse[s] = np.where(ok, es[best % len(es)], np.nan)
    return z, sg, lapse, n_tied


def run_marea_bl(win, px):
    """V10: V2 (V0's delays, the same grid and guards) with the lapse rates B_LAPSES."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win)
    tau_px = tau_px[px]
    h0 = LEV[0][w]
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    z, sigma, lapse, tied = (np.full(len(px), np.nan), np.full(len(px), np.nan), np.full(len(px), np.nan),
                             np.zeros(len(px), int))
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], sigma[cols], lapse[cols], tied[cols] = invert_bernoulli_lapse(Y[:, cols], C[:, cols],
                                                                               level_at(int(tau))[w], h0.min(), h0.max())
    fin = np.isfinite(z)
    share = ", ".join(f"{np.mean(lapse[fin] == e):.0%} e={e:g}" for e in B_LAPSES) if fin.any() else "no value"
    note = f"{lags_text(tau_bands)}; {share}; {int((tied[fin] > 1).sum()):,} of {int(fin.sum()):,} px on a tied plateau"
    return z, note, None, {"sigma": sigma, "lapse": lapse, "n_tied": tied}

In [ ]:
def brute_force_bl(wet, clear, h, lo, hi, lapses=B_LAPSES, tie_tol=B_TIE_TOL):
    """V10 of one pixel by plain loops over the grid and the lapse rates (the check of invert_bernoulli_lapse)."""
    wet = wet & clear
    if clear.sum() < B_MIN_LOOKS or not wet.any() or not (clear & ~wet).any():
        return np.nan
    grid, prof = np.linspace(lo, hi, 50), []
    for zk in grid:
        best = -np.inf
        for sk in marea.SG_GRID:
            for e in lapses:
                ll = 0.0
                for k in range(len(h)):
                    if clear[k]:
                        u = (h[k] - zk) / sk if wet[k] else -(h[k] - zk) / sk
                        ll += float(log_ndtr(u)) if e == 0 else float(np.log(e + (1 - 2 * e) * ndtr(u)))
                best = max(best, ll)
        prof.append(best)
    prof = np.array(prof)
    return float(np.median(grid[prof >= prof.max() - tie_tol]))


rng = np.random.default_rng(6)
hl = rng.uniform(-2.0, 2.0, 60)                        # 60 looks
low = np.argsort(hl)[:3]                               # the 3 lowest levels
Yl = np.stack([np.where(rng.random(60) < ndtr((hl - 0.3) / 0.25), 0.4, -0.3),    # a noisy pixel, z = 0.3
               np.where(hl > 0.5, 0.5, -0.2),                                   # complete separation at 0.5 ...
               np.where(hl > 0.5, 0.5, -0.2),                                   # ... and with 3 wet looks at low tide
               np.where(hl > 0.0, 0.5, -0.2)], axis=1)                          # 5 clear looks: no value
Yl[low, 2] = 0.3
Cl = rng.random((60, 4)) > 0.1
Cl[low, 2] = True
Cl[:, 3] = np.arange(60) < 5
zl, sl, el, ntl = invert_bernoulli_lapse(Yl, Cl, hl, hl.min(), hl.max())
with np.errstate(invalid="ignore"):
    zbl = [brute_force_bl(Yl[:, j] > 0, Cl[:, j], hl, hl.min(), hl.max()) for j in range(4)]
assert np.allclose(zl, zbl, rtol=0, atol=1e-12, equal_nan=True) and np.isnan(zl[3]), (zl, zbl)
z2, s2, _ = invert_bernoulli(Yl, Cl, hl, hl.min(), hl.max())
z0l, s0l, e0l, _ = invert_bernoulli_lapse(Yl, Cl, hl, hl.min(), hl.max(), lapses=(0.0,))
assert np.array_equal(z0l, z2, equal_nan=True) and np.array_equal(s0l, s2, equal_nan=True), "e = 0 alone is not V2"
print(f"invert_bernoulli_lapse = the brute-force maximum on 4 synthetic pixels: z = {np.round(zl, 3)}, "
      f"e = {el.tolist()}; with e = 0 alone it is invert_bernoulli (V2) bit for bit. The pixel with 3 wet looks at "
      f"low tide: V2 z = {z2[2]:.3f} m, V10 z = {zl[2]:.3f} m (e = {el[2]:g}); without them both give {zl[1]:.3f} m")

**The addendum's estimates** (`experiments/variants_addendum_2026-10-28.md`, declared before any window was computed
with them; diagnostics, run when `NB_METHODS` names them), each on V0's delay unless said otherwise:
**V11 MAREA-M**, the isotonic wet-probability read-off, no curve shape (`experiments/variants/monotone.py`), and
**V11d** with every delay 0; **V12 MAREA-P**, the depth-aware model NDWI = a + Φ(u)(b + c(h − z)), c ≥ 0, on V0's
inputs (`depth_model.py`); **V4 MAREA-H** and **V4-0**, the Huber fit (δ = 0.10) with weight 0.5 for the looks of SCL
class 7, its midpoint and its NDWI = 0 crossing (`huber.py`; SCL read once at the keep pixels of every scene of S, the
cube's for its scenes and the band-subset stores' before 2023); **V13a**, MAREA-B with no value where its 95 %
profile-likelihood interval is not bracketed by the grid (`interval.py`; its z is V2's, asserted bit for bit every
window); **V14**, the per-scene Otsu water threshold (`otsu.py`) as a factor on every wet/dry estimator: MAREA-B,
MAREA-BL and MAREA-M read NDWI − t_s, and LogiTide2DEM, sampled exceedance and the wet/dry bracket get it through
their inputs (`make_obs_otsu`), so that their own rule wet = NDWI > 0 is wet = NDWI > t_s; **V15 MAREA-S**, one
model and one search: the delay fit's own profiled z (its likelihood, clipping, grid and atoms; `single_pass.py`) at
its band lags, and **V15s** at V0's smoothed per-pixel delays. Section 8 has V13's interval checks, the error-coverage
curves of every estimate, the bootstrap of the judging rule and the elevation jump at the band limits (V15, V15s).

In [ ]:
from experiments.variants import (depth_model as v12, huber as v4, interval as v13, monotone as v11, otsu as v14,
                                  single_pass as v15)

ADD = {}            # the site's Otsu thresholds and shifted NDWI, and its SCL 7 looks: computed on first use


def otsu_ndwi():
    """V14: (Y_S - t_s per scene, t_s, the scenes that fell back to 0), once per site."""
    if "otsu" not in ADD:
        t, fell = v14.scene_thresholds(Y_S, C_S)
        ADD["otsu"] = (v14.shift_for_threshold(Y_S, t), np.asarray(t, float), np.asarray(fell, bool))
        print(f"V14: Otsu thresholds of the {len(t)} scenes of S: median {np.median(t):+.3f}, range "
              f"[{np.min(t):+.3f}, {np.max(t):+.3f}], {int(np.sum(fell))} fell back to 0", flush=True)
    return ADD["otsu"]


def scl7_at_keep():
    """V4: SCL == 7 at the keep pixels of every scene of S (scenes x keep, bool): the cube's SCL for its scenes, the
    band-subset stores' before 2023 (read as zarr_frames reads them)."""
    if "scl7" not in ADD:
        out = np.zeros((len(S_dates), len(keep)), bool)
        ds, bands, t_dim = pit.cube.open_cube(P["cache"])
        try:
            for k, d in enumerate(S_dates):
                if d in cube_pos:
                    out[k] = np.asarray(bands["SCL"].isel({t_dim: cube_pos[d]}).values).ravel()[keep] == 7
        finally:
            ds.close()
        old = [k for k, d in enumerate(S_dates) if d not in cube_pos]
        for year in sorted({S_dates[k][:4] for k in old}):
            ks = [k for k in old if S_dates[k][:4] == year]
            z = zc.open_store(zc.store_path(SITE, year, lte.LIGHT_ROOT), bands=["SCL"])
            try:
                zpos = {d: i for i, d in enumerate(zc.dates(z))}
                for a in range(0, len(ks), 16):
                    part = ks[a:a + 16]
                    scl = np.asarray(z["SCL"].isel({"t": [zpos[S_dates[k]] for k in part]}).values)
                    for j, k in enumerate(part):
                        out[k] = scl[j].ravel()[keep] == 7
            finally:
                z.close()
        ADD["scl7"] = out
        print(f"V4: SCL 7 on {np.mean(out[C_S]):.2%} of the clear looks of S at the keep pixels", flush=True)
    return ADD["scl7"]


def delay_groups(win, px, delay=True):
    """(the window's scenes, its band lags or None, the delay of each pixel of px, the window's lowest and highest
    level without delay)."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win) if delay else (None, np.zeros(len(keep)))
    h0 = LEV[0][w]
    return w, tau_bands, tau_px[px], h0.min(), h0.max()


def run_marea_m(win, px, delay=True, Ysrc=None):
    """V11 (V0's delays), V11d (every delay 0) and, with the shifted NDWI, V11 + Otsu."""
    w, tau_bands, tau_px, lo, hi = delay_groups(win, px, delay)
    Y, C = (Y_S if Ysrc is None else Ysrc)[np.ix_(w, px)], C_S[np.ix_(w, px)]
    z, cens = np.full(len(px), np.nan), np.zeros(len(px), int)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], cens[cols], _, _ = v11.invert_monotone(Y[:, cols], C[:, cols], level_at(int(tau))[w])
    fin = np.isfinite(z)
    note = (f"{lags_text(tau_bands)}; " if delay else "every delay 0; ") + \
        f"{int((cens[fin] != 0).sum()):,} of {int(fin.sum()):,} px censored"
    return z, note, None, {"censored": cens}


def run_marea_p(win, px):
    """V12 on V0's delays and inputs."""
    w, tau_bands, tau_px, lo, hi = delay_groups(win, px)
    Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
    z, c = np.full(len(px), np.nan), np.full(len(px), np.nan)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], _, c[cols] = v12.invert_depth(Y0[:, cols], C0[:, cols], level_at(int(tau))[w], lo, hi)
    fin = np.isfinite(z) & np.isfinite(c)
    note = (f"{lags_text(tau_bands)}; depth slope c: median {np.median(c[fin]):.3f} per m, "
            f"{np.mean(c[fin] > 0):.0%} of px > 0" if fin.any() else f"{lags_text(tau_bands)}; no value")
    return z, note, None, {"c": c}


HUBER = {}


def huber_fit(win, px):
    """V4's fit of the window at px, computed once: (midpoint z, NDWI = 0 crossing z0, iterations, note)."""
    key = (win["site"], win["n"])
    if key not in HUBER:
        w, tau_bands, tau_px, lo, hi = delay_groups(win, px)
        Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
        s7 = scl7_at_keep()[np.ix_(w, px)]
        z, z0, it = np.full(len(px), np.nan), np.full(len(px), np.nan), np.zeros(len(px), int)
        for tau in np.unique(tau_px):
            cols = np.flatnonzero(tau_px == tau)
            z[cols], z0[cols], _, _, _, it[cols] = v4.invert_huber(Y0[:, cols], C0[:, cols], level_at(int(tau))[w],
                                                                    lo, hi, scl7=s7[:, cols])
        fin = np.isfinite(z)
        HUBER[key] = (z, z0, it, f"{lags_text(tau_bands)}; iterations median "
                                 f"{int(np.median(it[fin])) if fin.any() else 0}, max {int(it.max()) if len(it) else 0}; "
                                 f"SCL 7 on {np.mean(s7[C0]):.1%} of the clear looks")
    return HUBER[key]


def run_marea_h(win, px, crossing=False):
    """V4 (the Huber fit's midpoint) and V4-0 (its NDWI = 0 crossing)."""
    z, z0, it, note = huber_fit(win, px)
    return (z0 if crossing else z), note, None, {"n_iter": it}


BINT = {}           # per window: MAREA-B's z and its 95 % interval at F (section 8)


def run_marea_b_abst(win, px):
    """V13a: MAREA-B with no value where its 95 % profile-likelihood interval is not bracketed by the grid."""
    w, tau_bands, tau_px, lo, hi = delay_groups(win, px)
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    b = {k: np.full(len(px), np.nan) for k in ("z", "z_lo", "z_hi")}
    br = np.zeros(len(px), bool)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z, _, zl, zh, ok = v13.bernoulli_interval(Y[:, cols], C[:, cols], level_at(int(tau))[w], lo, hi)
        b["z"][cols], b["z_lo"][cols], b["z_hi"][cols], br[cols] = z, zl, zh, ok
    BINT[(win["site"], win["n"])] = dict(b, bracketed=br)
    fin = np.isfinite(b["z"])
    note = f"{lags_text(tau_bands)}; {int((fin & br).sum()):,} of {int(fin.sum()):,} px with a bracketed interval"
    return (np.where(br, b["z"], np.nan), note, None,
            {"z_all": b["z"], "z_lo": b["z_lo"], "z_hi": b["z_hi"], "bracketed": br.astype(np.int8)})


def run_marea_b_otsu(win, px, lapse=False):
    """V14 on V2 (and on V10): their likelihoods on the wet/dry record wet = NDWI > t_s."""
    w, tau_bands, tau_px, lo, hi = delay_groups(win, px)
    Y, C = otsu_ndwi()[0][np.ix_(w, px)], C_S[np.ix_(w, px)]
    z = np.full(len(px), np.nan)
    fit = invert_bernoulli_lapse if lapse else invert_bernoulli
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols] = fit(Y[:, cols], C[:, cols], level_at(int(tau))[w], lo, hi)[0]
    return z, f"{lags_text(tau_bands)}; wet = NDWI > t_s (Otsu per scene)", None, {}


def make_obs_otsu(win, px, tau=0):
    """make_obs with the shifted NDWI (V14): wet = C & (Y - t_s > 0), NDWI > t_s, by the competitors' own rule."""
    w = win["scenes"]
    Y, C = otsu_ndwi()[0][np.ix_(w, px)], C_S[np.ix_(w, px)]
    with np.errstate(invalid="ignore"):
        wet = C & (Y > 0)
    return types.SimpleNamespace(Y=Y, C=C, W=wet, h=level_at(tau)[w], t=S_t[w], keep=keep[px],
                                 shape=(H, W), pixel_m=10.0, site=SITE)


def run_competitor_otsu(mid, win, px):
    """V14 on a wet/dry competitor (E4, E5c, E12): run_competitor's call with make_obs_otsu."""
    label, run, card = fm.METHODS[mid]
    cfg = {}
    if "level_series" in card.get("needs", []):
        start = S_t[win["scenes"][0]].normalize()
        cfg["level_series"] = LS.slice(start, pd.Timestamp(win["end"]) + pd.Timedelta(days=1))
    out = run(make_obs_otsu(win, px), cfg)
    return out["z"], (out["meta"].get("reason", "") + "; wet = NDWI > t_s (Otsu per scene)").lstrip("; "), None, {}


def run_marea_s(win, px, smooth=False):
    """V15 (the delay fit's own profiled z at its band lags, the bank read as te.m2a_rasch reads it) and V15s (the
    same at V0's smoothed per-pixel delays, the boundary level to the minute): the delay fit's likelihood, clipping,
    grid (z_puntos levels over the window's range +- 0.3 m) and atoms (sigma_perfil_m), every pixel."""
    w = win["scenes"]
    tau_bands, tau_px = window_delays(win)
    h0 = LEV[0][w]
    grid = np.linspace(h0.min() - 0.3, h0.max() + 0.3, M2["z_puntos"])
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    z = np.full(len(px), np.nan)
    if smooth:
        tp = tau_px[px]
        for tau in np.unique(tp):
            cols = np.flatnonzero(tp == tau)
            z[cols] = v15.invert_single(Y[:, cols], C[:, cols], level_at(int(tau))[w], grid, M2["sigma_perfil_m"])[0]
    else:
        bank = te.ShiftBank(marea.BANK_TAUS, [LEV[int(t)][w] for t in marea.BANK_TAUS])
        bands = band_of[px]
        for k in np.unique(bands[bands >= 0]):
            cols = np.flatnonzero(bands == k)
            z[cols] = v15.invert_single(Y[:, cols], C[:, cols], bank.at(tau_bands[k]), grid,
                                        M2["sigma_perfil_m"])[0]
    note = f"{lags_text(tau_bands)}; " + ("smoothed per-pixel delays" if smooth else "band lags")
    return z, note + "; the delay fit's likelihood and grid", None, {}

**The other estimates**, as notebook 01 runs them: *MAREA, no delay* (V0 with every delay 0, diagnostic); NIDEM
(`waterline_contour`, whole frames read from the cube, and before 2023 from the band-subset Zarr stores
`cubes_light/<site>/<year>.zarr`, not timed); DEA Intertidal 2.5.3 (DEA's own code in its side
interpreter, on every pixel of the extraction); LogiTide2DEM (E4), sampled exceedance (E5c) and Flood2Topo (E5a) of
`experiments/fixedcfg/methods.py`, per pixel on F; *LogiTide2DEM + MAREA delay* (diagnostic: one E4 call per delay
group, each with its levels read τ_px earlier). E4's per-pixel extras (its case: separated or fitted, its MSR) are
kept.

In [ ]:
def run_marea_nodelay(win, px):
    w = win["scenes"]
    h0 = LEV[0][w]
    Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
    z, _ = marea.invert_series(Y0, C0, h0, h0.min(), h0.max(), min_bracket=MAREA_MIN_BRACKET)
    return z, "every delay 0 (no lag fit)", None


def run_delayed(mid, win, px):
    """A per-pixel competitor with every pixel's levels read MAREA's delay earlier: one call per delay
    (notebook 01's run_delayed, with the window's delays fitted once)."""
    tau_bands, tau_px = window_delays(win)
    tau_px = tau_px[px]
    z = np.full(len(px), np.nan)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols] = fm.METHODS[mid][1](make_obs(win, px[cols], tau=int(tau)), {})["z"]
    return z, f"{lags_text(tau_bands)}; {len(np.unique(tau_px))} delay groups", None


cube_pos = {d: i for i, d in enumerate(cube.dates)}


STATIC = None       # the legacy cube's static no-data mask (H, W), set with the pre-2023 scenes (the frames checks)


def read_frames(dates):
    """NDWI and the common clear mask of whole scenes, (T, H, W): a scene of the cube is read from the cube, one
    scene at a time; a scene before the cube (pre-2023) from the band-subset Zarr store of its year (zarr_frames)."""
    ndwi, clear = np.empty((len(dates), H, W), np.float32), np.empty((len(dates), H, W), bool)
    ds, bands, t_dim = pit.cube.open_cube(P["cache"])
    try:
        for k, d in enumerate(dates):
            if d in cube_pos:
                at = {t_dim: cube_pos[d]}
                scl = bands["SCL"].isel(at).values
                ndwi[k], _ = wc.scene_ndwi(bands["B03"].isel(at).values, bands["B08"].isel(at).values, scl)
                clear[k] = np.isin(scl, wc.CLEAR)
    finally:
        ds.close()
    old = [k for k, d in enumerate(dates) if d not in cube_pos]
    if old:
        zarr_frames([dates[k] for k in old], ndwi, clear, old)
    return ndwi, clear


def zarr_frames(dates, ndwi, clear, at, root=lte.LIGHT_ROOT):
    """NDWI and the common clear mask of `dates` from the yearly Zarr stores <root>/<site>/<year>.zarr (B03, B08 and
    SCL as the backend delivered them, int16, on the legacy cube's frame), written into ndwi[at[i]], clear[at[i]].
    The fill -32768 is read as NaN, as xarray decodes the cube; then the cube's own scene_ndwi and clear mask. The
    pixels of STATIC (no data on every scene of the cube, data in the stores) get no data, the converter's rule.
    At most 16 scenes are read at a time (one chunk of the stores in time)."""
    for year in sorted({d[:4] for d in dates}):
        ks = [i for i, d in enumerate(dates) if d[:4] == year]
        z = zc.open_store(zc.store_path(SITE, year, root), bands=["B03", "B08", "SCL"])
        try:
            assert (z.sizes["y"], z.sizes["x"]) == (H, W), f"{root}/{SITE}/{year}.zarr is not on the cube's frame"
            zpos = {d: i for i, d in enumerate(zc.dates(z))}
            for a in range(0, len(ks), 16):
                part = ks[a:a + 16]
                sel = {"t": [zpos[dates[i]] for i in part]}
                b03, b08, scl = (np.asarray(z[b].isel(sel).values) for b in ("B03", "B08", "SCL"))
                for j, i in enumerate(part):
                    g, n = (np.where(v[j] == zc.FILL, np.float32(np.nan), v[j].astype(np.float32)) for v in (b03, b08))
                    ndwi[at[i]], _ = wc.scene_ndwi(g, n, scl[j])
                    clear[at[i]] = np.isin(scl[j], wc.CLEAR)
                    if STATIC is not None:
                        ndwi[at[i]][STATIC] = np.nan
                        clear[at[i]][STATIC] = False
        finally:
            z.close()


def run_nidem(win, px, frames):
    w = win["scenes"]
    j, edges, Z, U, sizes = wc.interval_assignment(LEV[0][w])
    wc.interval_coverage(wc.clear_counts_by_interval(C_S[w], Y_S[w], j)[0])   # pre-flight, on the pixels
    res = wc.nidem(wc.composites_from_arrays(*frames, j), Z, U)
    return res["filtered"].ravel()[keep[px]], f"{res['n_vertices']:,} contour vertices", None


def run_dea(win, px):
    out = dea.run(make_obs(win, np.arange(len(keep))), {"dea_tmp": DEA_TMP})
    return out["z"][px], f"{out['meta']['n_candidates']:,} candidate pixels", out["meta"].get("seconds_estimator")


def run_competitor(mid, win, px):
    """Notebook 01's run_competitor; the method's per-pixel extras (E4: msr, case, n_valid) are kept."""
    label, run, card = fm.METHODS[mid]
    cfg = {}
    if "level_series" in card.get("needs", []):
        start = S_t[win["scenes"][0]].normalize()
        cfg["level_series"] = LS.slice(start, pd.Timestamp(win["end"]) + pd.Timedelta(days=1))
    out = run(make_obs(win, px), cfg)
    extra = {k: v for k, v in out["meta"].get("extra", {}).items()
             if isinstance(v, np.ndarray) and v.shape == (len(px),)}
    return out["z"], out["meta"].get("reason", ""), None, extra


def run_method(mid, win, px, frames=None):
    """(z, note, the estimator's own seconds or None, per-pixel extras) of one estimate at the pixels px."""
    if mid == "MAREA":
        return run_marea(win, px)
    if mid == "MAREA_ndwi0":
        return run_marea_ndwi0(win, px)
    if mid == "MAREA_B":
        return run_marea_b(win, px)
    if mid == "MAREA_BL":
        return run_marea_bl(win, px)
    if mid == "MAREA_B_nodelay":
        return run_marea_b(win, px, delay=False)
    if mid == "MAREA_nodelay":
        return run_marea_nodelay(win, px) + ({},)
    if mid == "E4_delay":
        return run_delayed("E4", win, px) + ({},)
    if mid == "NIDEM":
        return run_nidem(win, px, frames) + ({},)
    if mid == "DEA":
        return run_dea(win, px) + ({},)
    if mid == "MAREA_M":                               # the addendum of 2026-10-28
        return run_marea_m(win, px)
    if mid == "MAREA_M_nodelay":
        return run_marea_m(win, px, delay=False)
    if mid == "MAREA_M_otsu":
        z, note, own, more = run_marea_m(win, px, Ysrc=otsu_ndwi()[0])
        return z, note + "; wet = NDWI > t_s (Otsu per scene)", own, more
    if mid == "MAREA_P":
        return run_marea_p(win, px)
    if mid in ("MAREA_H", "MAREA_H0"):
        return run_marea_h(win, px, crossing=mid == "MAREA_H0")
    if mid == "MAREA_B_abst":
        return run_marea_b_abst(win, px)
    if mid in ("MAREA_B_otsu", "MAREA_BL_otsu"):
        return run_marea_b_otsu(win, px, lapse=mid == "MAREA_BL_otsu")
    if mid in ("E4_otsu", "E5c_otsu", "E12_otsu"):
        return run_competitor_otsu(mid.removesuffix("_otsu"), win, px)
    if mid in ("MAREA_S", "MAREA_Ss"):
        return run_marea_s(win, px, smooth=mid == "MAREA_Ss")
    return run_competitor(mid, win, px)

**The pre-2023 frames: the checks.** The legacy cube has a static no-data block (no data on every one of its
scenes; `static_nodata` of the production reference map the extraction's JSON names, never a keep pixel) where the
Zarr stores have data: as the converter's screening does (its pre-registered rule), the Zarr frames get no data
there (`STATIC`), so NIDEM sees no observation the cube could not have. Then every Zarr store a window reads is on
the cube's frame (its x and y are the cube's, so the pixels come in the same order), and the reader gives the
cube's own arrays where both exist: on the screened scenes of 2023, the 2023 store (`cubes_light/<site>/2023.zarr`,
else the store the extraction's 2023 gate compared, `gate_2023` of its JSON) must give the clear mask of whole
frames, and their NDWI (NaN included) outside the static block, bit for bit as read from the cube, as that gate
found at the keep pixels on every 2023 scene. A difference stops the notebook unless the first author accepted the
2023 gate's differences (then it is reported).

In [ ]:
if len(x_dates):
    ref = side["reference_map"]                        # the production reference map and the static no-data mask
    assert lte.sha256_file(ref["path"]) == ref["sha256"], f"{ref['path']} is not the extraction's reference"
    with np.load(ref["path"]) as z:
        STATIC = z["static_nodata"].astype(bool)
    assert STATIC.shape == (H, W) and not STATIC.ravel()[keep].any()
    print(f"the legacy cube's static no-data ({ref['path']}): {int(STATIC.sum()):,} px, no keep pixel among them; "
          "the Zarr frames get no data there")
    with xr.open_dataset(P["cache"]) as ds:
        cube_x, cube_y = ds["x"].values, ds["y"].values
    gate = side["gate_2023"]
    light23 = zc.store_path(SITE, 2023, lte.LIGHT_ROOT)
    root23 = lte.LIGHT_ROOT if zc.is_complete(light23) else os.path.dirname(os.path.dirname(gate["store"]["path"]))
    years = sorted({d[:4] for d in x_dates})
    for year, root in [(y, lte.LIGHT_ROOT) for y in years] + [("2023", root23)]:
        with zc.open_store(zc.store_path(SITE, year, root), bands=["SCL"]) as z:
            assert np.array_equal(z["x"].values, cube_x) and np.array_equal(z["y"].values, cube_y), \
                f"{root}/{SITE}/{year}.zarr is not on the cube's frame"
    print(f"{lte.LIGHT_ROOT}/{SITE}/<year>.zarr of {years} and {root23}/{SITE}/2023.zarr: the cube's x and y")

    d23 = [d for d in S_dates if d[:4] == "2023"]      # the screened 2023 scenes: in the cube and in the store
    diff = Counter()
    for a in range(0, len(d23), 8):
        block = d23[a:a + 8]
        n_cube, c_cube = read_frames(block)            # from the cube
        n_zarr, c_zarr = np.empty_like(n_cube), np.empty_like(c_cube)
        zarr_frames(block, n_zarr, c_zarr, list(range(len(block))), root=root23)
        nan_cube, nan_zarr, out = np.isnan(n_cube), np.isnan(n_zarr), ~STATIC[None]   # NDWI: outside the static block
        diff["pixel-scenes"] += n_cube.size
        diff["NaN differs"] += int(((nan_cube != nan_zarr) & out).sum())
        diff["NDWI differs"] += int((~nan_cube & ~nan_zarr & out & (n_cube.view(np.uint32) != n_zarr.view(np.uint32)))
                                    .sum())
        diff["clear differs"] += int((c_cube != c_zarr).sum())                        # clear: the whole frame
    same = diff["NaN differs"] + diff["NDWI differs"] + diff["clear differs"] == 0
    print(f"{len(d23)} screened 2023 scenes, whole frames, {root23}/{SITE}/2023.zarr against the cube: "
          + ("the clear mask, and the NDWI (NaN included) outside the static block, bit for bit" if same
             else f"DIFFERENT {dict(diff)}"))
    print(f"the extraction's 2023 gate ({gate['store']['path']}, every 2023 scene at the keep pixels): "
          + ("passed" if gate.get("strict_ok") else f"differences accepted by {(gate.get('accepted') or {}).get('by')}"))
    if not same:
        assert gate.get("accepted"), "the Zarr frames are not the cube's, and no 2023 gate difference was accepted"
        print("  the 2023 gate's differences were accepted by the first author: NIDEM reads these frames (reported)")
else:
    print("no pre-2023 scene: NIDEM reads the cube only")

**Every window.** The window's delays are fitted first and timed as `delay_fit`; then every estimate is called and
timed with `time.perf_counter` (a failure is recorded with its reason and the others go on) and scored on the row's
pixels F by notebook 01's `scores` (RMSE, bias = median(p − t), centred RMSE, OLS slope of p on t, r, coverage). The
per-pixel estimates, extras, delays τ_px and clear-look counts are kept for the next sections. The CSV is rewritten
after every window.

In [ ]:
def scores(p, t):
    """RMSE, bias, centred RMSE, slope, r, the pixels scored and the coverage of p against t (same pixels)."""
    p, t = np.asarray(p, float), np.asarray(t, float)
    ok = np.isfinite(p) & np.isfinite(t)
    out = dict(n_px=int(ok.sum()), coverage=float(ok.mean()) if len(ok) else np.nan,
               rmse=np.nan, bias=np.nan, crmse=np.nan, slope=np.nan, r=np.nan)
    if ok.sum() >= 30:
        d = p[ok] - t[ok]
        pc, tc = p[ok] - np.median(p[ok]), t[ok] - np.median(t[ok])
        out.update(rmse=float(np.sqrt(np.mean(d ** 2))), bias=float(np.median(d)),
                   crmse=float(np.sqrt(np.mean((pc - tc) ** 2))), slope=float(np.polyfit(tc, pc, 1)[0]),
                   r=float(np.corrcoef(tc, pc)[0, 1]))
    return out


def run_window(win):
    """Every estimate on the window's images: the CSV rows, {id: elevation on the row's pixels F} and
    {id: its per-pixel extras}."""
    dates = S_dates[win["scenes"]]
    base = dict(site=win["site"], survey_year=win["year"], truth_date=win["end"], n=win["n"],
                available=win["available"], first_date=dates[0], last_date=dates[-1], n_F=len(win["px"]))
    t0 = time.perf_counter()
    tau_bands, _ = window_delays(win)
    rows = [dict(base, method="delay_fit", label="MAREA's delay fit (shared)", role="", status="ok",
                 reason=lags_text(tau_bands), seconds=time.perf_counter() - t0, seconds_own=np.nan)]
    print(f"{win['site']} n={win['n']} delay fit [{rows[0]['seconds']:.1f} s] {rows[0]['reason']}", flush=True)
    est, extra = {}, {}
    for mid in METHODS:
        r = dict(base, method=mid, label=LABELS.get(mid, mid), role=ROLE.get(mid, ""), status="ok", reason="",
                 seconds=np.nan, seconds_own=np.nan)
        try:
            frames = None
            if mid == "NIDEM":
                frames = read_frames(dates)            # the cube (before 2023 the Zarr stores) read, not timed
            t0 = time.perf_counter()
            z, note, own, more = run_method(mid, win, win["px"], frames)
            r.update(seconds=time.perf_counter() - t0, reason=note, seconds_own=np.nan if own is None else own)
            z = np.asarray(z, float)
            if np.isfinite(z).any():
                est[mid], extra[mid] = z, more
            else:
                r["status"] = "not computable"
        except (NotComputable, wc.EmptyIntervalError) as e:
            r.update(status="not computable", reason=str(e)[:300])
        except Exception as e:
            r.update(status="error", reason=repr(e)[:300])
        frames = None
        print(f"{win['site']} n={win['n']} {mid:15s} {r['status']} [{r['seconds']:.1f} s] {r['reason'][:90]}", flush=True)
        rows.append(r)
    if "MAREA" in est and "MAREA_ndwi0" in est:
        assert np.array_equal(extra["MAREA_ndwi0"]["z_mid"], est["MAREA"], equal_nan=True), "V1's fit is not V0's"
    if "MAREA_B" in est and "MAREA_B_abst" in est:
        assert np.array_equal(extra["MAREA_B_abst"]["z_all"], est["MAREA_B"], equal_nan=True), "V13's z is not V2's"
    return rows, est, extra

In [ ]:
SCORES, EST, EXTRA_PX, TAU, LOOKS = [], {}, {}, {}, {}
SENS = []                                              # v3: the scores at every q of the class mask
for win in windows:
    key = (win["site"], win["n"])
    if win["scenes"] is None:
        SCORES += [dict(site=win["site"], survey_year=win["year"], truth_date=win["end"], n=win["n"],
                        available=win["available"], method=m, label=LABELS.get(m, m), role=ROLE.get(m, ""),
                        status="not reachable",
                        reason=win.get("why") or f"{win['available']} screened images on or before {win['end']}")
                   for m in METHODS]
        continue
    rows, EST[key], EXTRA_PX[key] = run_window(win)
    if V3KEY:    # estimated once on every pixel with a clear look; scored at each q; kept at Q_CLASS from here on
        t_all = truth[win["px"]]
        for q, m in win["qmask"].items():
            common = m & np.isfinite(t_all)
            for mid, z in EST[key].items():
                if mid not in NOT_COMMON:
                    common &= np.isfinite(z)
            for mid, z in EST[key].items():
                for pixels, sel in (("own", m), ("common", common)):
                    SENS.append(dict(site=win["site"], n=win["n"], q=q, method=mid, label=LABELS.get(mid, mid),
                                     pixels=pixels, n_scored=int(sel.sum()), **scores(z[sel], t_all[sel])))
        main = win["qmask"][Q_CLASS]
        win["px_q0"], win["px"] = win["px"], win["px"][main]
        EST[key] = {mid: z[main] for mid, z in EST[key].items()}
        EXTRA_PX[key] = {mid: {k: (v[main] if isinstance(v, np.ndarray) and v.ndim >= 1 and len(v) == len(main)
                                   else v) for k, v in d.items()} for mid, d in EXTRA_PX[key].items()}
        for r in rows:
            r.update(truth_date=win["truth_date"], n_F=len(win["px"]), n_F_clear=len(win["px_q0"]), q=Q_CLASS,
                     half_width_days=win["half_width_days"], n_before=win["n_before"], n_after=win["n_after"],
                     centre=win["centre"], shift_days=win["shift_days"])
        pd.DataFrame(SENS).to_csv(f"{OUT}/{TAG}_class_sensitivity.csv", index=False)
    t_true = truth[win["px"]]
    for r in rows:
        if r["method"] in EST[key]:
            r.update(scores(EST[key][r["method"]], t_true))
    SCORES += rows
    w, px = win["scenes"], win["px"]
    TAU[key] = window_delays(win)[1][px]
    C_w = C_S[np.ix_(w, px)]
    with np.errstate(invalid="ignore"):
        LOOKS[key] = dict(n_clear=C_w.sum(axis=0), n_wet=(C_w & (Y_S[np.ix_(w, px)] > 0)).sum(axis=0))
    pd.DataFrame(SCORES).to_csv(f"{OUT}/{TAG}_scores.csv", index=False)
KEYS = list(EST)
print(f"-> {OUT}/{TAG}_scores.csv ({len(SCORES)} rows); windows computed: {KEYS}")
sc = pd.DataFrame(SCORES)
display(sc[sc.status == "ok"].pivot_table(index=["site", "n"], columns="method", values="rmse")
        .reindex(columns=[m for m in ORDER if m in set(sc.method)]).round(3))

## The maps

One GeoTIFF per method on the cube's grid; one figure, one colour scale (the 2nd to 98th percentile of every method's elevations).

In [ ]:
import rasterio

show = [m for m in ORDER if m in V3_METHODS]
MAPS_DIR = f"{OUT}/{TAG}_maps"
os.makedirs(MAPS_DIR, exist_ok=True)
summary = []
for key in KEYS:
    win = next(w for w in windows if (w["site"], w["n"]) == key)
    px = win["px"]
    est = {m: EST[key][m] for m in show if m in EST[key]}
    vals = np.concatenate([z[np.isfinite(z)] for z in est.values()])
    lo, hi = np.percentile(vals, [2, 98])
    rows_, cols_ = np.divmod(keep[px], W)
    r0, r1, c0, c1 = rows_.min(), rows_.max() + 1, cols_.min(), cols_.max() + 1
    ncol = min(5, len(est))
    nrow = int(np.ceil(len(est) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.7 * ncol, 2.7 * nrow * (r1 - r0) / (c1 - c0) + 0.9),
                             squeeze=False)
    for ax, (m, z) in zip(axes.flat, est.items()):
        r = np.full(H * W, np.nan, np.float32)
        r[keep[px]] = z
        with rasterio.open(f"{MAPS_DIR}/{key[0]}_n{key[1]}_{m}.tif", "w", driver="GTiff", height=H, width=W,
                           count=1, dtype="float32", crs=crs, transform=transform, nodata=np.nan,
                           compress="deflate") as dst:
            dst.write(r.reshape(H, W), 1)
        im = ax.imshow(r.reshape(H, W)[r0:r1, c0:c1], cmap="viridis", vmin=lo, vmax=hi, interpolation="nearest")
        fin = np.isfinite(z)
        ax.set_title(f"{LABELS.get(m, m)}\n{int(fin.sum()):,} px · median {np.nanmedian(z):+.2f} m", fontsize=7,
                     color=PAL["accent"] if m.startswith("MAREA") else PAL["ink"])
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        q = np.percentile(z[fin], [5, 50, 95]) if fin.any() else [np.nan] * 3
        summary.append(dict(site=key[0], n=key[1], method=m, label=LABELS.get(m, m), pixels=len(px),
                            with_value=int(fin.sum()), p5=q[0], median=q[1], p95=q[2]))
    for ax in list(axes.flat)[len(est):]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes, shrink=0.7, label="elevation (m, the boundary's datum)")
    d = S_dates[win["scenes"]]
    fig.suptitle(f"{V3KEY}: the elevation of each method, n = {key[1]} ({d[0]} → {d[-1]})", fontsize=9)
    fig.savefig(f"{OUT}/{TAG}_n{key[1]}_elevation_maps.png")
    plt.show()
summary = pd.DataFrame(summary)
summary.to_csv(f"{OUT}/{TAG}_elevation_summary.csv", index=False)
display(summary.set_index(["n", "label"])[["with_value", "p5", "median", "p95"]].round(3))
print(f"-> {MAPS_DIR}/ (one GeoTIFF per method), {OUT}/{TAG}_n*_elevation_maps.png, {OUT}/{TAG}_elevation_summary.csv")

## Provenance

In [ ]:
import platform, socket, subprocess
import scipy
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.rstrip()
dirty = [l for l in git("status", "--porcelain").splitlines() if not l.startswith("??")]
print("commit  ", git("rev-parse", "HEAD"), git("rev-parse", "--abbrev-ref", "HEAD"))
print(f"modified {len(dirty)} tracked file(s)" + "".join(f"\n  {l}" for l in dirty[:15]))
print("host    ", socket.gethostname(), "· OMP_NUM_THREADS", os.environ.get("OMP_NUM_THREADS"))
print("versions", f"python {platform.python_version()} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"xarray {xr.__version__} · scipy {scipy.__version__} · pyintertidal {pit.__version__}")